# Práctica Final LLM — RAG sobre transcripciones con metadatos

Buscador inteligente sobre un corpus de **transcripciones** que entiende preguntas en lenguaje natural **y respeta cómo están organizadas** (por carpeta, etiqueta y fecha). POC autónomo, en Python, **100 % local** (embeddings abiertos + LLM open-weights vía Ollama).

> **Motivación (alto nivel):** la idea nace de un proyecto personal —una app de notas de voz que organiza transcripciones en carpetas y con etiquetas—. Esta práctica es un POC independiente de ese problema de búsqueda.

**Corpus:** las **6 clases del módulo LLM** (~25 h de vídeo → ~203.000 palabras), en español, organizadas en carpetas (clases) y etiquetas (temas que cruzan varias clases).

Este notebook recorre el proyecto con los resultados ya ejecutados. El código vive en `src/` y la demo interactiva en `app.py`.

## 1. Configuración — cargamos el índice y el RAG
El índice ya está construido (`python src/01_ingest.py` → 1.394 chunks en ChromaDB). Aquí solo lo abrimos.

In [1]:
import sys, warnings, importlib.util
warnings.filterwarnings("ignore")
sys.path.insert(0, "src")
from rag_core import retrieve, format_citation

# src/03_rag.py tiene prefijo numérico (no importable como módulo) → lo cargamos por ruta
spec = importlib.util.spec_from_file_location("rag3", "src/03_rag.py")
rag3 = importlib.util.module_from_spec(spec); spec.loader.exec_module(rag3)
print("Índice y RAG cargados.")

Índice y RAG cargados.


## 2. El corpus y sus metadatos
Una fila por transcripción. Fíjate en que las **etiquetas cruzan clases** (p. ej. `evaluacion` aparece en Clases 3, 4 y 6; `rag` en 4 y 5): eso es lo que hace interesante el filtrado.

In [2]:
import pandas as pd
meta = pd.read_csv("data/metadata.csv")
meta[["carpeta", "area", "titulo", "fecha", "etiquetas", "n_palabras"]]

,carpeta,area,titulo,fecha,etiquetas,n_palabras
0,Clase 1,Fundamentos,"Ingeniería con Agentes de Código: Claude Code,...",2026-06-16,agentes-codigo;claude-code;sdd;skills;subagent...,32548
1,Clase 2,Fundamentos,"Fundamentos de LLMs: Transformers, Pretraining...",2026-06-17,tokens;embeddings;atencion;transformer;pretrai...,34470
2,Clase 3,Construcción,"LLMOps, Fine-tuning, Alignment y Evaluación",2026-06-18,fine-tuning;lora;qlora;distillation;alignment;...,36329
3,Clase 4,Construcción,"Frameworks, Embeddings, Vector Stores y RAG",2026-06-19,embeddings;vector-store;rag;chunking;retrieval...,33512
4,Clase 5,Producción,"Agentes: LangChain, LangGraph, MCP, ADK, A2A y...",2026-06-22,agentes;react;langchain;langgraph;mcp;adk;a2a;...,32372
5,Clase 6,Producción,"Producción: Observabilidad, Evaluación, Safety...",2026-06-23,observabilidad;langfuse;evaluacion;safety;red-...,33603


## 3. Recuperación: sin metadatos vs con metadatos
Misma pregunta, dos recuperaciones. **Relevante = el filtro debe enfocar los resultados** (menos mezcla entre clases).

In [3]:
def mostrar(hits):
    for d, s in hits:
        print(f"[{s:.3f}] {d.metadata['carpeta']:<8} · tags: {d.metadata['etiquetas'][:48]}")

q = "qué se dijo sobre la evaluación"
print("BASELINE (solo búsqueda semántica):")
mostrar(retrieve(q, k=4))
print("\nCON METADATOS (etiqueta = 'evaluacion'):")
mostrar(retrieve(q, k=4, etiqueta="evaluacion"))

BASELINE (solo búsqueda semántica):


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13607.66it/s]

[0.568] Clase 6  · tags: observabilidad;langfuse;evaluacion;safety;red-te
[0.566] Clase 5  · tags: agentes;react;langchain;langgraph;mcp;adk;a2a;n8
[0.531] Clase 4  · tags: embeddings;vector-store;rag;chunking;retrieval;r
[0.506] Clase 4  · tags: embeddings;vector-store;rag;chunking;retrieval;r

CON METADATOS (etiqueta = 'evaluacion'):
[0.568] Clase 6  · tags: observabilidad;langfuse;evaluacion;safety;red-te
[0.531] Clase 4  · tags: embeddings;vector-store;rag;chunking;retrieval;r
[0.506] Clase 4  · tags: embeddings;vector-store;rag;chunking;retrieval;r
[0.496] Clase 6  · tags: observabilidad;langfuse;evaluacion;safety;red-te


## 4. RAG end-to-end (generación local con Ollama)
Recupera (con filtros opcionales) y genera la respuesta con un LLM local, **citando la clase** y sin inventar.

In [4]:
for q, filtros in [
    ("¿Qué es un RAG y para qué sirve?", {"carpeta": "Clase 4"}),
    ("¿Qué diferencia hay entre LoRA y QLoRA?", {"carpeta": "Clase 3"}),
    ("¿Qué es el MCP (Model Context Protocol)?", {"etiqueta": "mcp"}),
]:
    texto, hits = rag3.answer(q, k=5, **filtros)
    print(f"❓ {q}   (filtros={filtros})\n")
    print(texto)
    print("\n📎 Fuentes:", ", ".join(d.metadata["carpeta"] for d, _ in hits))
    print("=" * 78)

❓ ¿Qué es un RAG y para qué sirve?   (filtros={'carpeta': 'Clase 4'})

RAG significa Retrieval-Augmented Generation, que se traduce como Generación Aumentada por Recuperación. Es una técnica utilizada en sistemas de asistencia con chatbots o generación de texto donde el modelo no solo genera respuestas basándose en su conocimiento interno, sino que también recupera información relevante de fuentes externas (como documentos, bases de datos, etc.) para mejorar la calidad y precisión de las respuestas.

El objetivo principal de RAG es permitir al sistema acceder a un mayor conjunto de información, lo que puede resultar en respuestas más precisas y relevantes. Esto se logra mediante el uso de un orquestador o agente que decide cuándo y cómo recuperar información externa para mejorar la respuesta generada por el modelo.

(Clase 4)

📎 Fuentes: Clase 4, Clase 4, Clase 4, Clase 4, Clase 4


❓ ¿Qué diferencia hay entre LoRA y QLoRA?   (filtros={'carpeta': 'Clase 3'})

Según el contexto proporcionado, QLoRA es una variante de LoRA. No se detalla mucho la diferencia específica entre LoRA y QLoRA en las transcripciones, pero se menciona que ambos son métodos para fine-tuning eficientes de modelos LLMs (Large Language Models).

(Clase 3)

No lo encuentro en las transcripciones.

📎 Fuentes: Clase 3, Clase 3, Clase 3, Clase 3, Clase 3


❓ ¿Qué es el MCP (Model Context Protocol)?   (filtros={'etiqueta': 'mcp'})

El MCP, o Model Context Protocol, es un protocolo que permite exponer herramientas de inteligencia artificial y otros recursos. Con MCP, no se necesita integrar manualmente la documentación como antes; ahora, se puede conectar directamente a las herramientas de IA para ejecutarlas y utilizar los recursos disponibles, incluyendo prompts y aplicaciones externas como Notion o bases de datos.

En resumen, el MCP facilita la interacción entre diferentes componentes del sistema, permitiendo que el host (la aplicación con el modelo) solicite y ejecute herramientas sin necesidad de leer documentación adicional. Este protocolo es especialmente útil para agilizar el desarrollo y la integración de nuevas funcionalidades.

(Clase 5)

📎 Fuentes: Clase 5, Clase 5, Clase 5, Clase 5, Clase 5


## 5. Evaluación — ¿mejora el filtrado por metadatos? (antes/después)
Golden set de 10 preguntas. **Relevante = chunk de la clase correcta.** Comparamos baseline vs filtrado por la etiqueta del tema.

In [5]:
spec2 = importlib.util.spec_from_file_location("eval4", "src/04_eval.py")
eval4 = importlib.util.module_from_spec(spec2); spec2.loader.exec_module(eval4)
eval4.run()


Golden set: 10 preguntas · k=5

Pregunta                                         BASE p@k META p@k BASE MRR META MRR
------------------------------------------------------------------------------------


¿Qué es el chunking y por qué se usa solapamie       0.80     1.00     1.00     1.00
¿Qué diferencia hay entre LoRA y QLoRA?              0.60     1.00     1.00     1.00
¿Qué es el mecanismo de atención en un Transfo       1.00     1.00     1.00     1.00
¿Qué es el MCP (Model Context Protocol) y para       0.80     1.00     0.50     1.00


¿Qué es RAG y qué pasos sigue?                       0.40     0.60     1.00     1.00
¿Qué es la observabilidad y qué hace Langfuse?       0.40     1.00     0.50     1.00
¿Qué es el fine-tuning y por qué causa olvido        0.60     1.00     0.50     1.00
¿Qué es un agente y en qué consiste el bucle R       1.00     1.00     1.00     1.00
¿Qué es el prompt injection y cómo se mitiga?        0.60     1.00     0.50     1.00
¿Qué es DPO en el alineamiento de modelos?           0.40     1.00     0.33     1.00
------------------------------------------------------------------------------------
MEDIA                                                0.66     0.96     0.73     1.00

Contaminación (nº medio de clases distintas en el top-5):
  BASELINE:   2.30 clases   |   CON METADATOS: 1.10 clases

Lectura: con el filtro por etiqueta sube la precisión@k y la respuesta
se concentra en la(s) clase(s) correcta(s) (menos contaminación).


## 6. Conclusiones

**Resultado principal:** filtrar por metadatos (la etiqueta del tema) sube la **precisión@5 de 0,66 → 0,96** y el **MRR de 0,73 → 1,00**, y reduce la contaminación entre clases (de 2,3 a 1,1 clases distintas por consulta). La tesis del proyecto —*los metadatos enfocan la recuperación*— queda demostrada.

**Decisiones de ingeniería:**
- **ES/ES, sin traducción:** corpus y preguntas en español; embeddings multilingües con buen español (`paraphrase-multilingual-mpnet-base-v2`). El objetivo es extraer información, no traducir.
- **LLM open-weights local** (Ollama / Qwen2.5-7B) para generación y juez. Sin claves ni coste.
- **Limpieza de términos del ASR (preprocesado, GIGO):** las transcripciones destrozaban los tecnicismos ("RAG"→"rack" 61 veces, "guardrail"→"WarRail"…). Sin corregirlo, la consulta limpia "RAG" no casaba con "rack" y el sistema se negaba a responder. `src/00_build_corpus.py` aplica un diccionario de correcciones antes de indexar.
- **Evaluación nativa en vez de RAGAS:** RAGAS 0.4.3 tiene un conflicto de imports con langchain 1.x; implementé las mismas ideas (precisión de contexto, fidelidad) de forma nativa.

**Limitaciones y mejoras futuras:** el ruido restante del ASR baja algo los scores; etiquetas que cruzan clases (p. ej. `rag`) no aíslan una sola clase. Mejoras: chunking más fino, expansión de consulta, reranking con cross-encoder, búsqueda híbrida (BM25 + semántica), y un `SelfQueryRetriever` que extraiga los filtros de la pregunta automáticamente.

**Demo interactiva:**
```bash
source .venv/bin/activate
python app.py        # http://127.0.0.1:7860 — pregunta y filtra por carpeta/etiqueta/fecha
```